In [1]:
import numpy as np
import os

# 정규화된 npy 샘플 확인
npy_dir = "/data/word_npy_norm"
files = os.listdir(npy_dir)
f = files[0]

kp = np.load(os.path.join(npy_dir, f))
print("shape:", kp.shape)
print("x 범위:", kp[..., 0].min(), "~", kp[..., 0].max())
print("y 범위:", kp[..., 1].min(), "~", kp[..., 1].max())
print("샘플값:\n", kp[0, :3])

shape: (120, 137, 3)
x 범위: 0.3635828 ~ 0.6881458
y 범위: 0.07944102 ~ 1.0
샘플값:
 [[0.51530415 0.20478055 1.        ]
 [0.5137901  0.39837405 1.        ]
 [0.42795727 0.39830184 1.        ]]


In [2]:
import numpy as np
import os
import re
from collections import Counter

npy_dir = "/data/word_npy_norm"
files = os.listdir(npy_dir)

labels = []
for f in files:
    match = re.search(r'WORD(\d+)', f)
    if match:
        labels.append(int(match.group(1)))

counter = Counter(labels)
print("고유 클래스 수:", len(counter))
print("클래스당 최소 샘플:", min(counter.values()))
print("클래스당 최대 샘플:", max(counter.values()))
print("클래스당 평균 샘플:", sum(counter.values()) / len(counter))
print("\n라벨 샘플 (앞 10개):", sorted(counter.keys())[:10])

고유 클래스 수: 3000
클래스당 최소 샘플: 13
클래스당 최대 샘플: 13
클래스당 평균 샘플: 13.0

라벨 샘플 (앞 10개): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [3]:
import numpy as np
import os

npy_dir = "/data/word_npy_norm"
files = sorted(os.listdir(npy_dir))

# 파일 3개 비교
for f in files[:3]:
    kp = np.load(os.path.join(npy_dir, f))
    print(f"{f}")
    print(f"  mean: {kp.mean():.4f} | std: {kp.std():.4f}")
    print(f"  첫프레임 첫좌표: {kp[0, 0]}")

NIA_SL_WORD0001_REAL01_F.npy
  mean: 0.6403 | std: 0.3075
  첫프레임 첫좌표: [0.49847552 0.19124445 1.        ]
NIA_SL_WORD0001_REAL02_F.npy
  mean: 0.6630 | std: 0.2980
  첫프레임 첫좌표: [0.50921404 0.2266861  1.        ]
NIA_SL_WORD0001_REAL03_F.npy
  mean: 0.6642 | std: 0.2988
  첫프레임 첫좌표: [0.50915104 0.22666667 1.        ]


In [4]:
# SignTransformer 대신 이거로 교체해서 테스트
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=411*120, num_classes=3000):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):              # (batch, 120, 137, 3)
        x = x.flatten(start_dim=1)     # (batch, 120*411)
        return self.net(x)

# train 함수에서 모델만 교체
model = SimpleClassifier().to(device)